# B2-019 — Session 3: Multi-Head Attention, Position, and Cost

*90 minutes.*

**Program layer:** Round 2 extension  
**Compute:** `compute.policy: cpu` · seed `20260808`  
**Qualified Book 1 prerequisites:** `book1:F1-scientific-python`, `book1:F3-matrices`, `book1:C6-pytorch`, `book1:C11-neural-training`  
**Remediation:** review the linked Book 1 units before continuing: [book1:F1-scientific-python](../../../../book1/units/F1-scientific-python/lesson.ipynb), [book1:F3-matrices](../../../../book1/units/F3-matrices/lesson.ipynb), [book1:C6-pytorch](../../../../book1/units/C6-pytorch/lesson.ipynb), [book1:C11-neural-training](../../../../book1/units/C11-neural-training/lesson.ipynb).


## 1. Projection and head dimensions

Let $X\in\mathbb R^{B\times n\times d}$ and $W_Q,W_K,W_V\in\mathbb R^{d\times d}$. The learned projections are $Q=XW_Q$, $K=XW_K$, and $V=XW_V$, each with shape $(B,n,d)$. With $h$ heads, require $h\mid d$ and set $d_h=d/h$. Reshape each projection to `(B,n,h,d_h)`, then transpose to `(B,h,n,d_h)` so every head owns a length-$n$ sequence.

**Worked example 1.** If `B=2`, `n=5`, `d=12`, and `h=3`, then $d_h=4$ and each projected tensor changes from `(2,5,12)` to `(2,3,5,4)` without changing its 120 scalar entries.

**Checkpoint 1A.** For $d=12,h=3$, state $d_h$.

**Checkpoint 1B.** Which permutation puts the head axis before sequence?

In [ ]:
import numpy as np
SEED = 20260808
def sinusoidal(length, width):
    positions = np.arange(length, dtype=np.float64)[:, None]
    frequencies = np.exp(-np.log(10000.0) * np.arange(0, width, 2) / width)
    table = np.zeros((length, width), dtype=np.float64)
    table[:, 0::2] = np.sin(positions * frequencies)
    table[:, 1::2] = np.cos(positions * frequencies[: table[:, 1::2].shape[1]])
    return table
assert np.array_equal(sinusoidal(3, 4)[0], [0., 1., 0., 1.])

## 2. Exact projection-to-output multiplication

For one head, write $Q_r=XW_Q^{(r)}$, $K_r=XW_K^{(r)}$, and $V_r=XW_V^{(r)}$, where each head projection has shape $(B,n,d_h)$. The score multiplication is $S_r=Q_rK_r^\top/\sqrt{d_h}$ with shape $(B,n,n)$. Normalize rows to obtain $A_r=\operatorname{softmax}(S_r)$, then multiply $H_r=A_rV_r$ to obtain shape $(B,n,d_h)$. Thus the contracting dimensions are explicit twice: $d_h$ in $Q_rK_r^\top$, then source length $n$ in $A_rV_r$.

**Worked example 2.** Take one batch, one head, $Q=[[1,0]]$, $K=[[1,0],[0,1]]$, and $V=[[2,0],[0,4]]$. Then $QK^\top=(1,0)$, $A=\operatorname{softmax}((1,0)/\sqrt2)$, and $AV=(2A_0,4A_1)$. The first multiply chooses source weights; the second mixes value features.

**Checkpoint 2A.** Why must Q and K share $d_h$?

**Checkpoint 2B.** Why must A and V share the source-length axis?

## 3. Independent heads and concatenation

Stacking heads gives `(B,h,n,d_h)`. Transpose to `(B,n,h,d_h)` and reshape the last two axes to `(B,n,d)` before the output projection $W_O\in\mathbb R^{d\times d}$. Heads are concatenated as features of the same token, never as extra sequence positions.

**Worked example 3.** Two one-dimensional heads returning 3 and -2 concatenate to `(3,-2)` at the same token.

**Checkpoint 3A.** Why must concatenation use the feature direction rather than sequence?

**Checkpoint 3B.** What shape enters the output projection?

In [ ]:
B, n, d, h = 2, 5, 12, 3
dh = d // h
heads = np.zeros((B, h, n, dh))
concatenated = heads.transpose(0, 2, 1, 3).reshape(B, n, d)
assert concatenated.shape == (B, n, d)

## 4. Sinusoidal positional encoding

Without a position signal, permuting input rows permutes output rows in the same way: self-attention sees content and pairwise matches but no absolute order. For zero-based position $p$ and even coordinate $2i$, use $\sin(p/10000^{2i/d})$; for odd coordinate $2i+1$, use $\cos(p/10000^{2i/d})$. Add the `(L,d)` table to numeric token inputs before the Q/K/V projections.

**Worked example 4.** At $p=0$, all sine coordinates are 0 and all cosine coordinates are 1. At later positions, high-frequency coordinates change faster than low-frequency coordinates.

**Checkpoint 4A.** Why does pure self-attention need an added order signal?

**Checkpoint 4B.** State the positional table shape for maximum length $L$.

## 5. Exact time and score-memory cost

The three input projections and one output projection each scale as $Bnd^2$, so projection work is $\Theta(Bnd^2)$. Per head, both $QK^\top$ and $AV$ cost $Bn^2d_h$; multiplying by $h$ and using $hd_h=d$ gives $\Theta(Bn^2d)$ across all heads. Materialized scores or weights require exactly $Bhn^2$ scalars. Changing $h$ at fixed $d$ leaves the leading attention arithmetic unchanged but increases score storage linearly.

**Worked example 5.** For `B=2`, `h=4`, and `n=100`, one score tensor has `2*4*100*100 = 80,000` scalars. Doubling `n` creates 320,000 scalars; doubling `d` alone does not change this score count.

**Checkpoint 5A.** If $n$ doubles, by what factor does score memory change?

**Checkpoint 5B.** If $d$ doubles at fixed $n$, what happens to the attention-product term?

## 6. Common pitfalls and forward route

**Common pitfalls.** Broken: apply attention directly to reshaped X and silently omit $W_Q,W_K,W_V$. Fix: trace $XW_Q$, $XW_K$, and $XW_V$ before splitting heads. Broken: concatenate on the sequence axis, multiplying length by $h$. Fix: restore `(B,n,h,d_h)` first. Broken: count only one head's score matrix. Fix: include $h$ in memory while simplifying arithmetic with $hd_h=d$.

**Exam connections.** Practice p16 follows only after this exact projection/attention multiply derivation: separate projection work, the two attention multiplications, and materialized score memory. Round 2 questions distinguish arithmetic from storage and demand exact dimensions.

**Going deeper.** Session 4 turns the shape ledger into a tested module and tiny training loop.

Checkpoint answers: 1A 4; 1B transpose head and sequence axes; 2A the dot-product axis must match; 2B each weight multiplies its associated value row; 3A heads describe features for the same token; 3B $(B,n,d)$; 4A attention alone is permutation equivariant; 4B $(L,d)$; 5A fourfold; 5B twofold.